# 6 — Lipid tail splay

All-atom CHARMM36 binary bilayer (DOPC/DOPA, DOPC/DOPG, DOPC/DLPG, DOPC/DLPA).

Angle between the two acyl chains of each lipid (first tail carbon → terminal carbon vectors). Single frame gives the across-molecule distribution.


## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy scipy matplotlib
%matplotlib inline
import os, re, itertools
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

## Config — edit these

You have **no trajectory**, so `TRAJ = None` loads the single `.gro`
frame (a **snapshot**). When you have an `.xtc`/`.trr`, set `TRAJ` to it
and every cell below averages over frames automatically.

In [ ]:
TOPOLOGY = "md.tpr"        # topology (has bonds + masses)
STRUCTURE= "md.gro"        # single-frame structure (.gro or .pdb)
TRAJ     = None            # e.g. "md_center.xtc" when you have one; None -> use STRUCTURE

LIPIDS   = ["DOPC", "DOPA"]        # the TWO lipids in THIS system
HEAD     = "P"                     # headgroup / leaflet marker
STEP     = 1                       # frame stride (only matters with a trajectory)
OUTDIR   = "results"; os.makedirs(OUTDIR, exist_ok=True)

## Load

In [ ]:
u = mda.Universe(TOPOLOGY, TRAJ) if TRAJ else mda.Universe(TOPOLOGY, STRUCTURE)
has_bonds = len(getattr(u, "bonds", [])) > 0
present   = [r for r in LIPIDS if len(u.select_atoms(f"resname {r}")) > 0]
nframes   = len(u.trajectory)
print(f"{len(u.atoms)} atoms, {nframes} frame(s), bonds={has_bonds}")
print("Lipids present:", present)
if nframes == 1:
    print("NOTE: single frame -> snapshot only (no time averaging).")

## Splay angle distribution

In [ ]:
def carbons(res, prefix):
    pat=re.compile(r"^"+prefix+r"(\d+)$")
    hits=[(int(m.group(1)),at) for at in res.atoms if (m:=pat.match(at.name)) and int(m.group(1))>=2]
    return sorted(hits,key=lambda t:t[0])
def ends(res,prefix):
    cs=carbons(res,prefix); return (cs[0][1],cs[-1][1]) if len(cs)>=2 else None
cache={}
for r in present:
    cache[r]=[]
    for res in u.select_atoms(f"resname {r}").residues:
        e2,e3=ends(res,"C2"),ends(res,"C3")
        if e2 and e3: cache[r].append((e2[0].index,e2[1].index,e3[0].index,e3[1].index))
angles={r:[] for r in present}
for ts in u.trajectory[::STEP]:
    pos=u.atoms.positions
    for r in present:
        for i2,j2,i3,j3 in cache[r]:
            v2=pos[j2]-pos[i2]; v1=pos[j3]-pos[i3]; n2=np.linalg.norm(v2); n1=np.linalg.norm(v1)
            if n1 and n2: angles[r].append(np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2),-1,1))))
plt.figure(figsize=(7,4))
for r in present:
    a=np.array(angles[r])
    if a.size:
        print(f"{r}: <splay> = {a.mean():.1f} +/- {a.std():.1f} deg")
        plt.hist(a,bins=40,density=True,histtype="step",label=r)
plt.xlabel("tail splay angle (deg)"); plt.ylabel("P(theta)"); plt.legend()
plt.grid(alpha=.3); plt.title("Tail splay"); plt.show()